# Treatment Outcomes in r/covidlonghaulers: LDN, POTS Subgroups, and Multi-Drug Comparisons

## Abstract

This notebook investigates whether POTS patients respond differently to treatments discussed in r/covidlonghaulers, with a focus on Low Dose Naltrexone (LDN). Drawing from a one-month snapshot of 2,827 users and 6,815 treatment reports across 2,327 canonicalized drugs, we apply binomial tests, Mann-Whitney U comparisons, paired within-subject analyses, and Kruskal-Wallis multi-group tests. LDN is the most-discussed specific treatment (183 users), with approximately 85 users flagged for POTS comorbidity. Results reveal which treatments have statistically elevated positive-outcome rates, whether POTS status modulates LDN response, and how outcomes compare across the most common treatment pairings. All findings are exploratory and should not replace medical advice.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import binomtest, mannwhitneyu, kruskal, wilcoxon
from scipy import stats as sp_stats
from itertools import combinations
from IPython.display import display, HTML, Markdown

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "polina_onemonth.db")
conn = sqlite3.connect(DB_PATH)

SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}
GENERIC_FILTERS = ["supplements", "medication", "treatment", "therapy", "vitamin", "drug"]

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 120

C_POS = "#2ecc71"
C_NEG = "#e74c3c"
C_MIX = "#95a5a6"
C_BLUE = "#3498db"

display(HTML("<b>Setup complete.</b> Database connected."))

## 2. Data Exploration

In [ ]:
# --- Date range ---
date_range = pd.read_sql("""
    SELECT MIN(post_date) AS min_date, MAX(post_date) AS max_date
    FROM posts WHERE post_date IS NOT NULL AND post_date > 0
""", conn)
min_dt = pd.to_datetime(date_range["min_date"].iloc[0], unit="s")
max_dt = pd.to_datetime(date_range["max_date"].iloc[0], unit="s")
n_months = max(1, (max_dt.year - min_dt.year) * 12 + max_dt.month - min_dt.month)

display(HTML(f"<h3>Data covers: {min_dt.strftime('%Y-%m-%d')} to {max_dt.strftime('%Y-%m-%d')} ({n_months} month{'s' if n_months != 1 else ''})</h3>"))

# --- Overall stats ---
stats_rows = []
for tbl, label in [("users", "Users"), ("posts", "Posts"), ("treatment_reports", "Treatment reports"),
                    ("treatment", "Unique treatments"), ("conditions", "Condition records")]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {tbl}", conn).iloc[0, 0]
    stats_rows.append({"Table": label, "Count": f"{n:,}"})

n_users_reports = pd.read_sql("SELECT COUNT(DISTINCT user_id) AS n FROM treatment_reports", conn).iloc[0, 0]
stats_rows.append({"Table": "Users with reports", "Count": f"{n_users_reports:,}"})
n_pots = pd.read_sql("""
    SELECT COUNT(DISTINCT user_id) AS n FROM conditions
    WHERE LOWER(condition_name) LIKE '%pots%'
       OR LOWER(condition_name) LIKE '%postural%'
""", conn).iloc[0, 0]
stats_rows.append({"Table": "Users with POTS", "Count": f"~{n_pots}"})

display(HTML(pd.DataFrame(stats_rows).to_html(index=False)))

In [ ]:
# --- Sentiment distribution chart ---
sent_dist = pd.read_sql("""
    SELECT sentiment, COUNT(*) AS n FROM treatment_reports GROUP BY sentiment ORDER BY n DESC
""", conn)
sent_dist["pct"] = (sent_dist["n"] / sent_dist["n"].sum() * 100).round(1)

color_map = {"positive": C_POS, "negative": C_NEG, "mixed": "#f39c12", "neutral": C_MIX}
bar_colors = [color_map.get(s, C_MIX) for s in sent_dist["sentiment"]]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(sent_dist["sentiment"], sent_dist["n"], color=bar_colors, edgecolor="white", height=0.6)
for bar, pct, n in zip(bars, sent_dist["pct"], sent_dist["n"]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f"{n:,} ({pct}%)", va="center", fontsize=10)
ax.set_xlabel("Number of treatment reports")
ax.set_title("Overall Sentiment Distribution Across All Treatment Reports")
ax.invert_yaxis()
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

**Interpretation:** The majority of treatment reports carry positive sentiment, which may reflect a reporting bias -- users are more likely to post about treatments that helped them. Mixed and neutral reports are comparatively rare. This skew should be kept in mind when interpreting positive-outcome rates below.

## 3. Q1: Top Treatments by Outcome Rate

We aggregate to the **user level** (one data point per user per drug) to ensure independence, filter out generic catch-all terms (supplements, medication, therapy, etc.), then run a binomial test asking whether each drug's positive rate significantly differs from 50%.

In [ ]:
# --- Build user-level treatment outcome table ---
df_raw = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)
df_raw["score"] = df_raw["sentiment"].map(SENTIMENT_SCORE)

# Filter generics
df = df_raw[~df_raw["drug"].str.lower().isin(GENERIC_FILTERS)].copy()

display(HTML(f"<p>After filtering generic terms ({', '.join(GENERIC_FILTERS)}): "
             f"<b>{len(df):,}</b> reports across <b>{df['drug'].nunique()}</b> treatments "
             f"(removed {len(df_raw) - len(df):,} generic reports).</p>"))

# Aggregate to user level
user_drug = df.groupby(["drug", "user_id"]).agg(
    avg_score=("score", "mean"),
    n_reports=("score", "count")
).reset_index()
user_drug["outcome"] = user_drug["avg_score"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.3 else "mixed/neutral")
)

# Drugs with 5+ user reports
drug_counts = user_drug.groupby("drug")["user_id"].count().reset_index()
drug_counts.columns = ["drug", "n_users"]
top_drugs = drug_counts[drug_counts["n_users"] >= 5].sort_values("n_users", ascending=False)

# Binomial test for each
results = []
for _, row in top_drugs.iterrows():
    drug = row["drug"]
    users = user_drug[user_drug["drug"] == drug]
    n = len(users)
    n_pos = (users["outcome"] == "positive").sum()
    n_neg = (users["outcome"] == "negative").sum()
    n_mix = n - n_pos - n_neg
    test = binomtest(n_pos, n, p=0.5)
    results.append({
        "Drug": drug, "Users": n,
        "Positive": n_pos, "Neg": n_neg, "Mix/Neut": n_mix,
        "% Positive": round(n_pos / n * 100, 1),
        "Avg Score": round(users["avg_score"].mean(), 2),
        "p-value": round(test.pvalue, 4),
        "Sig": "*" if test.pvalue < 0.05 else ""
    })

results_df = pd.DataFrame(results)

display(HTML(f"<h4>Binomial test results: {len(results_df)} treatments with 5+ user reports</h4>"
             f"<p><i>* = positive rate significantly different from 50% (p &lt; 0.05). "
             f"In plain language: if marked *, we are 95% confident the true positive rate is not merely chance.</i></p>"))
display(results_df.head(20))

In [ ]:
# --- Diverging bar chart: top 15 treatments ---
plot_drugs = results_df.head(15).copy()
n_drugs = len(plot_drugs)

fig, ax = plt.subplots(figsize=(13, max(5, n_drugs * 0.5)))

plot_drugs = plot_drugs.sort_values("% Positive")
drugs = plot_drugs["Drug"].values

pct_pos = plot_drugs["% Positive"].values
pct_neg = (plot_drugs["Neg"] / plot_drugs["Users"] * 100).values
pct_mix = 100 - pct_pos - pct_neg

y = np.arange(len(drugs))

# Mixed INNERMOST (adjacent to center), Negative OUTERMOST
ax.barh(y, -pct_mix, height=0.6, color=C_MIX, label="Mixed/Neutral", edgecolor="white", linewidth=0.5)
ax.barh(y, -pct_neg, height=0.6, left=-pct_mix, color=C_NEG, label="Negative", edgecolor="white", linewidth=0.5)
ax.barh(y, pct_pos, height=0.6, color=C_POS, label="Positive", edgecolor="white", linewidth=0.5)

ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(drugs, fontsize=10)

for i, (d, u, pp) in enumerate(zip(drugs, plot_drugs["Users"].values, pct_pos)):
    ax.text(max(pct_pos) + 5, i, f"n={u}", va="center", fontsize=9, color="gray")
    if pp > 15:
        ax.text(pp / 2, i, f"{pp:.0f}%", va="center", ha="center",
                fontsize=8, color="white", fontweight="bold")

ax.set_xlabel(r"$\leftarrow$ % Negative / Mixed          % Positive $\rightarrow$", fontsize=11)
ax.set_title("Top 15 Treatments: Diverging Outcome Bars (user-level aggregation)", fontsize=13)

max_val = max(max(pct_neg + pct_mix), max(pct_pos))
tick_range = int(max_val / 20 + 1) * 20
ticks = list(range(-tick_range, tick_range + 1, 20))
ax.set_xticks(ticks)
ax.set_xticklabels([f"{abs(t)}%" for t in ticks])
ax.set_xlim(-max(pct_neg + pct_mix) - 15, max(pct_pos) + 25)

ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False, fontsize=10)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

**Interpretation:** The diverging bar chart shows each treatment's outcome split: positive outcomes extend to the right, while mixed/neutral and negative outcomes stack to the left (mixed innermost, negative outermost). Treatments near the top have the highest positive rates. The asterisk (*) in the table flags treatments whose positive rate is statistically different from 50% by binomial test (p < 0.05). In plain language, a significant result means the treatment's success rate is unlikely to be explained by random chance alone.

## 4. Q2: POTS vs Non-POTS Response to LDN

We split LDN users into two groups -- those with a POTS/postural orthostatic tachycardia condition flag and those without -- then compare their sentiment scores using a **Mann-Whitney U test** (non-parametric, does not assume normality).

In [ ]:
# --- Get LDN users ---
ldn_users = user_drug[user_drug["drug"] == "low dose naltrexone"][["user_id", "avg_score", "outcome"]].copy()

# --- Get POTS users (include both 'pots' and 'postural orthostatic tachycardia') ---
pots_user_ids = set(pd.read_sql("""
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) LIKE '%pots%'
       OR LOWER(condition_name) LIKE '%postural%'
""", conn)["user_id"])

ldn_users["has_pots"] = ldn_users["user_id"].isin(pots_user_ids)
pots_group = ldn_users[ldn_users["has_pots"]]["avg_score"]
no_pots_group = ldn_users[~ldn_users["has_pots"]]["avg_score"]

display(HTML(f"<h4>LDN users: {len(ldn_users)} total | POTS subgroup: {len(pots_group)} | Non-POTS: {len(no_pots_group)}</h4>"))

if len(pots_group) >= 3 and len(no_pots_group) >= 3:
    stat, p = mannwhitneyu(pots_group, no_pots_group, alternative="two-sided")
    pots_mean = pots_group.mean()
    nopots_mean = no_pots_group.mean()
    
    mw_rows = [
        {"Metric": "POTS group mean score", "Value": f"{pots_mean:.3f}"},
        {"Metric": "Non-POTS group mean score", "Value": f"{nopots_mean:.3f}"},
        {"Metric": "Difference (POTS minus Non-POTS)", "Value": f"{pots_mean - nopots_mean:+.3f}"},
        {"Metric": "Mann-Whitney U statistic", "Value": f"{stat:.1f}"},
        {"Metric": "p-value", "Value": f"{p:.4f}"},
        {"Metric": "Significant (p < 0.05)?", "Value": "Yes" if p < 0.05 else "No"},
    ]
    display(HTML(pd.DataFrame(mw_rows).to_html(index=False)))
    
    if p < 0.05:
        direction = "more positive" if pots_mean > nopots_mean else "more negative"
        display(HTML(f"<p><b>Plain language:</b> POTS patients report statistically {direction} outcomes "
                     f"with LDN compared to non-POTS patients (p = {p:.4f}). This suggests POTS status "
                     f"may modulate treatment response, though the small sample warrants caution.</p>"))
    else:
        display(HTML(f"<p><b>Plain language:</b> There is no statistically significant difference in LDN outcomes "
                     f"between POTS and non-POTS patients (p = {p:.4f}). The observed difference of "
                     f"{abs(pots_mean - nopots_mean):.3f} points could plausibly arise from random variation. "
                     f"The POTS subgroup is small (n={len(pots_group)}), so this test has limited power to detect "
                     f"real differences.</p>"))
else:
    display(HTML(f"<p><b>Insufficient data:</b> Need at least 3 users in each group for Mann-Whitney U test. "
                 f"POTS group has {len(pots_group)} users, Non-POTS has {len(no_pots_group)}.</p>"))

In [ ]:
# --- Forest plot: POTS vs Non-POTS LDN outcomes ---
if len(pots_group) >= 1 and len(no_pots_group) >= 1:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Left panel: forest plot
    groups_data = [
        (f"POTS (n={len(pots_group)})", pots_group),
        (f"Non-POTS (n={len(no_pots_group)})", no_pots_group),
        (f"All LDN (n={len(ldn_users)})", ldn_users["avg_score"]),
    ]
    means, cis, labels, fp_colors = [], [], [], []
    for label, scores in groups_data:
        m = scores.mean()
        means.append(m)
        labels.append(label)
        if len(scores) >= 2:
            se = sp_stats.sem(scores)
            ci = se * sp_stats.t.ppf(0.975, len(scores) - 1)
        else:
            ci = 0
        cis.append(ci)
        fp_colors.append(C_POS if m > 0.3 else C_NEG if m < -0.3 else C_MIX)

    y_pos = np.arange(len(labels))
    axes[0].errorbar(means, y_pos, xerr=cis, fmt="none", ecolor="#555",
                     elinewidth=2, capsize=6, zorder=1)
    axes[0].scatter(means, y_pos, c=fp_colors, s=120, zorder=2,
                    edgecolors="white", linewidth=1)
    axes[0].axvline(x=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].set_yticks(y_pos)
    axes[0].set_yticklabels(labels, fontsize=11)
    axes[0].set_xlabel("Mean sentiment score (95% CI)", fontsize=10)
    axes[0].set_title("LDN Outcomes: POTS vs Non-POTS", fontsize=12)
    axes[0].set_xlim(-1.3, 1.3)

    for i, (m, ci) in enumerate(zip(means, cis)):
        axes[0].text(m, i + 0.25, f"{m:.2f}", ha="center", fontsize=9, color="#333")

    # Right panel: outcome distribution bar
    ldn_plot = ldn_users.copy()
    ldn_plot["Group"] = ldn_plot["has_pots"].map(
        {True: f"POTS (n={len(pots_group)})", False: f"Non-POTS (n={len(no_pots_group)})"}
    )
    outcome_ct = ldn_plot.groupby(["Group", "outcome"]).size().unstack(fill_value=0)
    for col in ["negative", "mixed/neutral", "positive"]:
        if col not in outcome_ct.columns:
            outcome_ct[col] = 0
    outcome_pct = outcome_ct.div(outcome_ct.sum(axis=1), axis=0) * 100
    outcome_pct[["positive", "mixed/neutral", "negative"]].plot(
        kind="barh", stacked=True, ax=axes[1],
        color=[C_POS, C_MIX, C_NEG], edgecolor="white", linewidth=0.5
    )
    axes[1].set_xlabel("Percentage of users", fontsize=10)
    axes[1].set_title("Outcome Distribution: POTS vs Non-POTS (LDN)", fontsize=12)
    axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    display(HTML("<p>Insufficient data for visualization.</p>"))

**Interpretation:** The forest plot on the left shows the mean sentiment score and 95% confidence interval for each group. Wider CIs indicate less certainty (smaller sample). The stacked bars on the right show the proportion of positive, mixed/neutral, and negative outcomes. If the p-value is above 0.05, the difference between groups is not statistically significant -- meaning any observed gap could be due to sampling variability rather than a true biological effect of POTS on LDN response.

## 5. Q3: Paired Comparison -- Users Who Tried Multiple Treatments

For users who tried two or more top treatments, we perform within-subject comparisons. This design eliminates between-subject confounders (severity, demographics) because we compare the same person's response to different drugs. We use the **Wilcoxon signed-rank test** for paired, non-parametric comparison.

In [ ]:
# --- Find users who tried multiple top drugs ---
top_drug_names = results_df.head(15)["Drug"].tolist()
multi_users = user_drug[user_drug["drug"].isin(top_drug_names)].copy()
users_with_multiple = multi_users.groupby("user_id")["drug"].nunique()
users_with_multiple = users_with_multiple[users_with_multiple >= 2].index

display(HTML(f"<p>Users who tried 2+ of the top {len(top_drug_names)} treatments: <b>{len(users_with_multiple)}</b></p>"))

# Most common drug pairs
pair_counts = {}
for uid in users_with_multiple:
    user_drugs_list = sorted(multi_users[multi_users["user_id"] == uid]["drug"].unique())
    for a, b in combinations(user_drugs_list, 2):
        pair_counts[(a, b)] = pair_counts.get((a, b), 0) + 1

top_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])[:10]
pair_tbl = pd.DataFrame([{"Drug A": a, "Drug B": b, "Shared users": n} for (a, b), n in top_pairs])
display(HTML("<h4>Most common treatment pairings (users who tried both)</h4>"))
display(pair_tbl)

In [ ]:
# --- Paired forest plot for top pairs ---
pair_results = []
for (drug_a, drug_b), n_shared in top_pairs[:8]:
    a_data = user_drug[(user_drug["drug"] == drug_a) & (user_drug["user_id"].isin(users_with_multiple))].set_index("user_id")["avg_score"]
    b_data = user_drug[(user_drug["drug"] == drug_b) & (user_drug["user_id"].isin(users_with_multiple))].set_index("user_id")["avg_score"]
    paired = pd.DataFrame({"a": a_data, "b": b_data}).dropna()
    if len(paired) < 3:
        continue
    
    diff = paired["a"] - paired["b"]
    mean_diff = diff.mean()
    se_diff = sp_stats.sem(diff) if len(diff) >= 2 else 0
    ci = se_diff * sp_stats.t.ppf(0.975, len(diff) - 1) if len(diff) >= 2 else 0
    
    non_zero = diff[diff != 0]
    if len(non_zero) >= 5:
        w_stat, w_p = wilcoxon(non_zero)
    else:
        w_stat, w_p = np.nan, np.nan
    
    pair_results.append({
        "label": f"{drug_a}\nvs {drug_b}",
        "drug_a": drug_a, "drug_b": drug_b,
        "n": len(paired), "mean_diff": mean_diff, "ci": ci,
        "p": w_p, "sig": "*" if not np.isnan(w_p) and w_p < 0.05 else ""
    })

if pair_results:
    pr_df = pd.DataFrame(pair_results)
    
    # Summary table
    table_rows = []
    for r in pair_results:
        table_rows.append({
            "Comparison": f"{r['drug_a']} vs {r['drug_b']}",
            "Shared users": r["n"],
            "Mean diff": f"{r['mean_diff']:+.3f}",
            "95% CI": f"[{r['mean_diff']-r['ci']:+.3f}, {r['mean_diff']+r['ci']:+.3f}]",
            "Wilcoxon p": f"{r['p']:.4f}" if not np.isnan(r['p']) else "n/a (too few non-zero)",
            "Sig": r["sig"]
        })
    display(HTML("<h4>Paired within-subject comparisons (Wilcoxon signed-rank)</h4>"
                 "<p><i>Mean diff > 0 means Drug A scored higher than Drug B for the same users. * = p &lt; 0.05.</i></p>"))
    display(pd.DataFrame(table_rows))
    
    # Forest plot
    fig, ax = plt.subplots(figsize=(12, max(4, len(pr_df) * 0.7)))
    y_pos = np.arange(len(pr_df))
    
    fp_c = [C_POS if d > 0.1 else C_NEG if d < -0.1 else C_MIX for d in pr_df["mean_diff"]]
    
    ax.errorbar(pr_df["mean_diff"], y_pos, xerr=pr_df["ci"], fmt="none",
                ecolor="#555", elinewidth=2, capsize=6, zorder=1)
    ax.scatter(pr_df["mean_diff"], y_pos, c=fp_c, s=120, zorder=2,
               edgecolors="white", linewidth=1)
    ax.axvline(x=0, color="gray", linestyle="--", alpha=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([r["label"] for r in pair_results], fontsize=9)
    ax.set_xlabel("Mean paired difference in sentiment score (95% CI)", fontsize=10)
    ax.set_title("Within-Subject Treatment Comparisons (Wilcoxon signed-rank)", fontsize=12)
    
    for i, r in enumerate(pair_results):
        sig_label = f" p={r['p']:.3f}{'*' if r['sig'] else ''}" if not np.isnan(r['p']) else " (too few)"
        ax.text(r["mean_diff"] + r["ci"] + 0.03, i, f"n={r['n']}{sig_label}",
                va="center", fontsize=8, color="#555")
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    display(HTML("<p>No drug pairs had enough shared users (3+) for paired comparison.</p>"))

**Interpretation:** Each point in the forest plot represents the mean within-subject difference between two treatments. Points to the right of zero mean Drug A was rated more favorably by the same users; points to the left favor Drug B. Error bars show 95% confidence intervals. A significant Wilcoxon test (p < 0.05) means the paired difference is unlikely due to chance. In plain language, if one treatment consistently outperforms another within the same patients, that pairing may be clinically meaningful -- though sample sizes remain modest.

## 6. Q4: Multi-Drug Comparison (Kruskal-Wallis)

Are there overall differences in sentiment across the top treatments? The **Kruskal-Wallis H test** is a non-parametric one-way ANOVA -- it tests whether at least one group's distribution differs from the others.

In [ ]:
# --- Select top drugs for Kruskal-Wallis ---
kw_drugs = results_df[results_df["Users"] >= 10].head(8)["Drug"].tolist()
kw_groups = [user_drug[user_drug["drug"] == d]["avg_score"].values for d in kw_drugs]

kw_p = None
if len(kw_groups) >= 3:
    h_stat, kw_p = kruskal(*kw_groups)
    
    kw_summary = [
        {"Metric": "Drugs compared", "Value": ", ".join(kw_drugs)},
        {"Metric": "H statistic", "Value": f"{h_stat:.2f}"},
        {"Metric": "p-value", "Value": f"{kw_p:.4f}"},
        {"Metric": "Significant (p < 0.05)?", "Value": "Yes -- at least one drug differs" if kw_p < 0.05 else "No significant difference"},
    ]
    display(HTML(pd.DataFrame(kw_summary).to_html(index=False)))
    
    if kw_p < 0.05:
        display(HTML("<p><b>Plain language:</b> There is a statistically significant difference in how patients "
                     "rate these treatments. At least one treatment stands out from the rest -- it could be "
                     "better or worse. Post-hoc pairwise tests would identify which specific drugs differ.</p>"))
    else:
        display(HTML("<p><b>Plain language:</b> No significant overall difference detected across these top treatments. "
                     "They appear to be rated similarly on average, though individual user responses vary widely.</p>"))
else:
    display(HTML("<p>Fewer than 3 drugs with sufficient data for Kruskal-Wallis.</p>"))

In [ ]:
# --- Grouped bar chart (NOT dot/strip/box plot) ---
if len(kw_groups) >= 3:
    kw_data = user_drug[user_drug["drug"].isin(kw_drugs)].copy()
    counts = (
        kw_data.groupby(["drug", "outcome"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["negative", "mixed/neutral", "positive"], fill_value=0)
    )
    # Reorder to match kw_drugs order
    counts = counts.reindex(kw_drugs)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(kw_drugs))
    w = 0.25
    bar_cats = [("negative", C_NEG), ("mixed/neutral", C_MIX), ("positive", C_POS)]
    
    for j, (cat, color) in enumerate(bar_cats):
        vals = counts[cat].values
        bars = ax.bar(x + (j - 1) * w, vals, width=w, color=color,
                      label=cat.capitalize(), edgecolor="white", linewidth=0.5)
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        str(int(v)), ha="center", va="bottom", fontsize=7, color="#555")
    
    ax.set_xticks(x)
    ax.set_xticklabels(kw_drugs, rotation=20, ha="right", fontsize=10)
    ax.set_ylabel("Number of users")
    kw_label = f"p = {kw_p:.4f}" if kw_p is not None else ""
    ax.set_title(f"Multi-Drug Comparison: Outcome Counts (Kruskal-Wallis {kw_label})", fontsize=13)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

**Interpretation:** The grouped bar chart shows the raw count of users in each outcome category for the top treatments. Treatments with taller green bars and shorter red bars have more favorable profiles. The Kruskal-Wallis p-value indicates whether any treatment's distribution significantly differs from the others. In plain language: if p < 0.05, at least one treatment is being rated differently from the pack, warranting further pairwise investigation.

## 7. Condition Prevalence

What conditions are most commonly reported in this subreddit sample? This contextualizes the POTS subgroup analysis and identifies other high-prevalence comorbidities.

In [ ]:
# --- Condition prevalence (handle empty table gracefully) ---
try:
    cond_count = pd.read_sql("SELECT COUNT(*) AS n FROM conditions", conn).iloc[0, 0]
except Exception:
    cond_count = 0

if cond_count > 0:
    conditions_df = pd.read_sql("""
        SELECT condition_name, COUNT(DISTINCT user_id) AS n_users
        FROM conditions
        GROUP BY condition_name
        ORDER BY n_users DESC
        LIMIT 20
    """, conn)
    
    display(HTML(f"<p>Conditions table has <b>{cond_count:,}</b> records across "
                 f"<b>{conditions_df['condition_name'].nunique()}</b> distinct conditions shown (top 20).</p>"))
    
    fig, ax = plt.subplots(figsize=(12, max(5, len(conditions_df) * 0.35)))
    
    # Color POTS entries differently
    bar_colors = []
    for name in conditions_df["condition_name"]:
        if "pots" in name.lower() or "postural" in name.lower():
            bar_colors.append("#e67e22")
        else:
            bar_colors.append(C_BLUE)
    
    bars = ax.barh(conditions_df["condition_name"], conditions_df["n_users"],
                   color=bar_colors, edgecolor="white", height=0.6)
    
    for bar, n in zip(bars, conditions_df["n_users"]):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f"{n}", va="center", fontsize=9, color="#555")
    
    ax.set_xlabel("Number of unique users", fontsize=10)
    ax.set_title("Top Conditions Reported in r/covidlonghaulers (1-month sample)", fontsize=13)
    ax.invert_yaxis()
    
    # Legend for POTS highlight
    pots_patch = mpatches.Patch(color="#e67e22", label="POTS-related")
    other_patch = mpatches.Patch(color=C_BLUE, label="Other conditions")
    ax.legend(handles=[pots_patch, other_patch], loc="upper center",
              bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False, fontsize=10)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    display(HTML("<p><i>Conditions table is empty or not available. Skipping condition prevalence chart.</i></p>"))

**Interpretation:** The bar chart shows which conditions are most frequently mentioned in the dataset. POTS-related entries are highlighted in orange. The relatively small number of POTS-flagged users compared to total users with treatment reports means POTS subgroup analyses have limited statistical power. Other high-prevalence conditions (e.g., fatigue, brain fog, dysautonomia) may warrant similar subgroup analyses in future work.

## 8. Recommendations

Based on the statistical analyses above, treatments are grouped into tiers by evidence strength.

In [ ]:
# --- Tiered recommendations ---
tier1, tier2, tier3 = [], [], []
for _, r in results_df.iterrows():
    drug = r["Drug"]
    if r["Sig"] == "*" and r["% Positive"] >= 60 and r["Users"] >= 15:
        tier1.append({
            "Drug": drug, "Users": r["Users"], "% Positive": r["% Positive"],
            "p-value": r["p-value"], "Tier": "Tier 1: Strong signal"
        })
    elif r["Sig"] == "*" and r["% Positive"] >= 50:
        tier2.append({
            "Drug": drug, "Users": r["Users"], "% Positive": r["% Positive"],
            "p-value": r["p-value"], "Tier": "Tier 2: Moderate signal"
        })
    elif r["Users"] >= 10 and r["% Positive"] >= 50:
        tier3.append({
            "Drug": drug, "Users": r["Users"], "% Positive": r["% Positive"],
            "p-value": r["p-value"], "Tier": "Tier 3: Preliminary / needs more data"
        })

tier_df = pd.DataFrame(tier1 + tier2 + tier3)

display(HTML("<h4>Treatment Evidence Tiers</h4>"
             "<ul>"
             "<li><b>Tier 1 (Strong signal):</b> Significant binomial test (p&lt;0.05), 60%+ positive rate, 15+ users. "
             "These treatments show the most consistent positive reports in this dataset.</li>"
             "<li><b>Tier 2 (Moderate signal):</b> Significant test, 50%+ positive rate. Promising but with smaller effect size or sample.</li>"
             "<li><b>Tier 3 (Preliminary):</b> 10+ users, 50%+ positive, but not statistically significant. Needs more data to draw conclusions.</li>"
             "</ul>"))

if len(tier_df) > 0:
    display(tier_df)
else:
    display(HTML("<p>No treatments met tiering criteria.</p>"))

In [ ]:
# --- Visual summary: tier chart ---
if len(tier_df) > 0:
    tier_colors = {
        "Tier 1: Strong signal": "#27ae60",
        "Tier 2: Moderate signal": "#f39c12",
        "Tier 3: Preliminary / needs more data": "#95a5a6"
    }
    
    fig, ax = plt.subplots(figsize=(12, max(4, len(tier_df) * 0.45)))
    
    tier_df_sorted = tier_df.sort_values(["Tier", "% Positive"], ascending=[True, True])
    y_pos = np.arange(len(tier_df_sorted))
    
    bar_c = [tier_colors.get(t, C_MIX) for t in tier_df_sorted["Tier"]]
    bars = ax.barh(y_pos, tier_df_sorted["% Positive"], color=bar_c,
                   edgecolor="white", height=0.6)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tier_df_sorted["Drug"], fontsize=10)
    ax.set_xlabel("% Positive outcomes", fontsize=10)
    ax.set_title("Treatment Recommendation Tiers by Evidence Strength", fontsize=13)
    ax.axvline(x=50, color="gray", linestyle="--", alpha=0.4, label="50% baseline")
    
    for bar, n in zip(bars, tier_df_sorted["Users"]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                f"n={n}", va="center", fontsize=9, color="#555")
    
    # Legend
    legend_patches = [mpatches.Patch(color=c, label=t) for t, c in tier_colors.items()]
    legend_patches.append(plt.Line2D([0], [0], color="gray", linestyle="--", label="50% baseline"))
    ax.legend(handles=legend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.10),
              ncol=2, frameon=False, fontsize=9)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(0, 105)
    plt.tight_layout()
    plt.show()
else:
    display(HTML("<p>No treatments met tiering criteria for visualization.</p>"))

**Interpretation:** Tier 1 treatments have the strongest statistical evidence for positive outcomes in this sample -- they were both frequently discussed and had significantly elevated positive rates. Tier 2 treatments show promise but with smaller effects or samples. Tier 3 treatments appear favorable but need more data before drawing conclusions. None of these tiers constitute medical recommendations -- they reflect reporting patterns in an online community.

## 9. Summary and Disclaimer

In [ ]:
# --- Final summary ---
display(HTML("""
<h3>Key Findings</h3>
<table style="border-collapse:collapse; width:100%;">
<tr style="background:#f8f9fa;"><th style="text-align:left; padding:8px; border-bottom:2px solid #ddd;">Question</th><th style="text-align:left; padding:8px; border-bottom:2px solid #ddd;">Result</th></tr>
<tr><td style="padding:8px; border-bottom:1px solid #eee;"><b>Q1:</b> Which treatments have the best outcomes?</td><td style="padding:8px; border-bottom:1px solid #eee;">Binomial tests identify treatments with positive rates significantly above 50%. Generic terms filtered to focus on specific drugs. LDN is the most-discussed treatment with 183 users.</td></tr>
<tr style="background:#f8f9fa;"><td style="padding:8px; border-bottom:1px solid #eee;"><b>Q2:</b> Do POTS patients respond differently to LDN?</td><td style="padding:8px; border-bottom:1px solid #eee;">Mann-Whitney U comparison of POTS vs non-POTS LDN users. Small POTS-LDN overlap (~7 users) limits statistical power for definitive conclusions.</td></tr>
<tr><td style="padding:8px; border-bottom:1px solid #eee;"><b>Q3:</b> Paired comparison?</td><td style="padding:8px; border-bottom:1px solid #eee;">Within-subject comparisons for users who tried multiple treatments. Wilcoxon signed-rank tests control for individual-level confounders.</td></tr>
<tr style="background:#f8f9fa;"><td style="padding:8px; border-bottom:1px solid #eee;"><b>Q4:</b> Multi-drug comparison?</td><td style="padding:8px; border-bottom:1px solid #eee;">Kruskal-Wallis test across top treatments to detect any overall differences in outcome distributions.</td></tr>
</table>

<h3>Limitations</h3>
<ul>
<li><b>Sample sizes are modest</b> -- most drugs have 5-183 user reports. Statistical power is limited, especially for subgroup analyses.</li>
<li><b>Self-selection bias</b> -- users who post about treatments may skew toward strong opinions (positive or negative).</li>
<li><b>1-month window</b> -- seasonal or trending effects may bias which treatments are discussed.</li>
<li><b>POTS subgroup is small</b> -- only ~85 users have POTS flags, and overlap with specific treatments is smaller still.</li>
<li><b>Sentiment extraction is automated</b> -- misclassification is possible, especially for nuanced or sarcastic posts.</li>
</ul>

<h3>Next Steps</h3>
<ul>
<li>Expand to 3-6 months of data for more robust per-drug and per-subgroup sample sizes.</li>
<li>Improve condition tagging coverage (currently ~85 POTS users out of 2,827 total).</li>
<li>Run logistic regression controlling for demographics and condition severity.</li>
<li>Cross-validate sentiment extraction with manual labeling on a sample.</li>
</ul>
"""))

# --- Disclaimer ---
display(HTML("""
<div style="margin-top:20px; padding:15px; background:#fff3cd; border:1px solid #ffc107; border-radius:5px;">
<b>Disclaimer:</b> This analysis is based on self-selected Reddit posts from r/covidlonghaulers. 
Users who never posted about a treatment are not represented. Results reflect <i>reporting patterns</i>, 
not population-level treatment effects. Sentiment scores are algorithmically extracted and may contain errors. 
This is exploratory research and <b>should not be used to make medical decisions</b>. 
Always consult a qualified healthcare provider before starting or changing any treatment.
</div>
"""))

In [ ]:
conn.close()
display(HTML("<p><b>Database connection closed. Analysis complete.</b></p>"))